# Student lab — a diffusion policy from scratch

Train a conditional diffusion model to generate an entire 2-D motion chunk. The agent must move from the left edge to a commanded goal while avoiding a circular obstacle. Demonstrations go **above** or **below** the obstacle, so the data are deliberately multimodal.

By the end you will have implemented the three operations that make a diffusion policy work:

1. forward noising, $q(x_t\mid x_0)$;
2. denoising-score training, $\lVert \epsilon-\epsilon_\theta(x_t,t,c)\rVert^2$;
3. iterative reverse sampling, $x_T\rightarrow x_0$.

Suggested pacing: 15 min setup and intuition · 35 min implementation and training · 20 min evaluation · 15 min stress tests and discussion.

## 0 · Environment

This notebook uses PyTorch and Matplotlib. It runs comfortably on CPU; a notebook GPU shortens the training loop.

This is deliberately **not** a miniature VLA. We remove vision, language, and robot hardware so that the only remaining question is the one the exercise is meant to answer: how can an action model represent several valid futures without averaging them together?

In [ ]:
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

N_POINTS = 24                  # one trajectory/action chunk
X_GRID = torch.linspace(-1.0, 1.0, N_POINTS, device=DEVICE)

plt.rcParams.update({
    "figure.figsize": (9, 4.6),
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## 1 · Make a tiny multimodal imitation-learning dataset

Every condition contains the desired terminal height and obstacle radius, $c=[y_{goal}, r]$. For the same condition, an expert may choose either a positive or negative arch. We diffuse only the $y$ coordinates; the $x$ coordinates are fixed and known.

In [ ]:
def make_batch(batch_size):
    s = torch.linspace(0.0, 1.0, N_POINTS, device=DEVICE).view(1, -1)
    goal_y = torch.empty(batch_size, 1, device=DEVICE).uniform_(-0.25, 0.25)
    radius = torch.empty(batch_size, 1, device=DEVICE).uniform_(0.18, 0.28)
    mode = torch.where(
        torch.rand(batch_size, 1, device=DEVICE) < 0.5,
        -torch.ones(batch_size, 1, device=DEVICE),
        torch.ones(batch_size, 1, device=DEVICE),
    )

    baseline = s * goal_y
    amplitude = radius + 0.24 + 0.05 * torch.randn(batch_size, 1, device=DEVICE)
    smooth_noise = 0.02 * torch.randn(batch_size, N_POINTS, device=DEVICE) * torch.sin(math.pi * s)
    y = baseline + mode * amplitude * torch.sin(math.pi * s) + smooth_noise
    y[:, 0] = 0.0
    y[:, -1] = goal_y[:, 0]

    cond = torch.cat([goal_y, radius], dim=1)
    return y, cond, mode[:, 0]


def draw_obstacle(ax, radius, color="#d97706", alpha=0.22):
    obstacle = plt.Circle((0.0, 0.0), radius, color=color, alpha=alpha)
    ax.add_patch(obstacle)
    ax.add_patch(plt.Circle((0.0, 0.0), radius, fill=False, color=color, lw=2))


y_demo, cond_demo, mode_demo = make_batch(16)
fig, ax = plt.subplots()
for y in y_demo.detach().cpu():
    ax.plot(X_GRID.cpu(), y, color="#0f766e", alpha=0.45, lw=2)
draw_obstacle(ax, float(cond_demo[:, 1].mean()))
ax.scatter([-1, 1], [0, 0], s=60, c=["#111827", "#2563eb"], zorder=5)
ax.set(xlim=(-1.08, 1.08), ylim=(-0.9, 0.9), xlabel="x", ylabel="trajectory y")
ax.set_title("Expert demonstrations: two valid modes")
plt.show()

### Why squared-error regression is the wrong baseline

The conditional mean of equally likely upper and lower demonstrations is approximately a straight line. That line runs through the obstacle—the average of two good answers is a bad answer.

In [ ]:
goal_y = 0.0
radius = 0.23
s = torch.linspace(0.0, 1.0, N_POINTS)
upper = s * goal_y + (radius + 0.30) * torch.sin(math.pi * s)
lower = s * goal_y - (radius + 0.30) * torch.sin(math.pi * s)
mean_path = 0.5 * (upper + lower)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for path, color in [(upper, "#0f766e"), (lower, "#2563eb")]:
    axes[0].plot(torch.linspace(-1, 1, N_POINTS), path, color=color, lw=3)
draw_obstacle(axes[0], radius)
axes[0].set_title("Two expert modes")

axes[1].plot(torch.linspace(-1, 1, N_POINTS), mean_path, color="#dc2626", lw=3)
draw_obstacle(axes[1], radius)
axes[1].set_title("MSE prediction: collision")
for ax in axes:
    ax.set(xlim=(-1.05, 1.05), ylim=(-0.8, 0.8))
plt.show()

## 2 · Define the forward diffusion process

For a randomly chosen timestep $t$,

$$x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon,\qquad \epsilon\sim\mathcal N(0,I).$$

Complete **q_sample**. This is the only forward-process equation the training loop needs.

In [ ]:
T = 50
betas = torch.linspace(1e-4, 0.10, T, device=DEVICE)
alphas = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)
alpha_bar_prev = torch.cat([torch.ones(1, device=DEVICE), alpha_bar[:-1]])
posterior_variance = betas * (1.0 - alpha_bar_prev) / (1.0 - alpha_bar).clamp_min(1e-8)


def q_sample(x0, t, noise):
    """Sample x_t ~ q(x_t | x_0)."""
    # TODO 1: implement the closed-form forward process.
    # Hint: gather alpha_bar[t], reshape it to [batch, 1], then mix x0 and noise.
    raise NotImplementedError("Implement q_sample")

In [ ]:
# Quick check: the final timestep should be much noisier than the first.
x0_check, _, _ = make_batch(128)
fixed_noise = torch.randn_like(x0_check)
early = q_sample(x0_check, torch.zeros(128, dtype=torch.long, device=DEVICE), fixed_noise)
late = q_sample(x0_check, torch.full((128,), T - 1, dtype=torch.long, device=DEVICE), fixed_noise)
assert early.shape == x0_check.shape
assert F.mse_loss(late, x0_check) > 10 * F.mse_loss(early, x0_check)
print("q_sample check passed")

## 3 · Build the conditional denoiser

The model sees the noisy action chunk $x_t$, a sinusoidal embedding of $t$, and the task condition $c$. A small MLP is enough for this toy problem; real diffusion policies commonly use temporal convolutional or transformer backbones.

In [ ]:
def timestep_embedding(t, dim=32):
    half = dim // 2
    frequencies = torch.exp(
        -math.log(10_000) * torch.arange(half, device=t.device) / max(half - 1, 1)
    )
    angles = (t.float() / max(T - 1, 1)).unsqueeze(1) * frequencies.unsqueeze(0) * 1000
    return torch.cat([torch.sin(angles), torch.cos(angles)], dim=1)


class Denoiser(nn.Module):
    def __init__(self, trajectory_dim=N_POINTS, cond_dim=2, time_dim=32, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(trajectory_dim + cond_dim + time_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, trajectory_dim),
        )

    def forward(self, xt, t, cond):
        features = torch.cat([xt, timestep_embedding(t), cond], dim=1)
        return self.net(features)


model = Denoiser().to(DEVICE)
sum(parameter.numel() for parameter in model.parameters())

## 4 · Implement the training objective

Training is ordinary supervised regression once we manufacture the target noise. Complete **diffusion_loss**, then train the denoiser.

In [ ]:
def diffusion_loss(model, x0, cond):
    # TODO 2: sample a timestep and Gaussian noise, construct x_t,
    # predict the noise, and return mean-squared error.
    raise NotImplementedError("Implement diffusion_loss")

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-5)
loss_history = []
TRAIN_STEPS = 4_000

model.train()
for step in range(TRAIN_STEPS):
    x0, cond, _ = make_batch(256)
    loss = diffusion_loss(model, x0, cond)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    loss_history.append(float(loss.detach()))

    if (step + 1) % 250 == 0:
        recent = np.mean(loss_history[-100:])
        print(f"step {step + 1:4d} · loss {recent:.4f}")

plt.plot(loss_history, color="#d97706", alpha=0.65)
plt.yscale("log")
plt.xlabel("optimization step")
plt.ylabel("noise-prediction MSE")
plt.title("Training curve")
plt.show()

## 5 · Implement reverse diffusion

At inference, start with Gaussian noise and repeatedly apply the learned denoiser. Complete one DDPM reverse step in **p_sample**.

The sampler below also projects the known first and last waypoint after every step. In robotics language, those are hard boundary conditions.

In [ ]:
@torch.no_grad()
def p_sample(model, xt, t, cond):
    """One DDPM reverse step: x_t -> x_{t-1}."""
    # TODO 3: predict noise, compute the DDPM posterior mean,
    # and add posterior noise unless t == 0.
    raise NotImplementedError("Implement p_sample")

In [ ]:
@torch.no_grad()
def sample_trajectories(model, cond):
    model.eval()
    batch = cond.shape[0]
    xt = torch.randn(batch, N_POINTS, device=DEVICE)
    snapshots = {}

    for step in reversed(range(T)):
        t = torch.full((batch,), step, dtype=torch.long, device=DEVICE)
        xt = p_sample(model, xt, t, cond)
        xt[:, 0] = 0.0
        xt[:, -1] = cond[:, 0]
        if step in {T - 1, 35, 20, 5, 0}:
            snapshots[step] = xt.detach().cpu()
    return xt, snapshots


cond_test = torch.tensor([[0.0, 0.23]] * 64, device=DEVICE)
samples, snapshots = sample_trajectories(model, cond_test)

fig, ax = plt.subplots(figsize=(9, 5))
for y in samples.detach().cpu():
    ax.plot(torch.linspace(-1, 1, N_POINTS), y, color="#0f766e", alpha=0.20, lw=2)
draw_obstacle(ax, 0.23)
ax.set(xlim=(-1.05, 1.05), ylim=(-0.85, 0.85), xlabel="x", ylabel="generated y")
ax.set_title("Conditional diffusion samples")
plt.show()

## 6 · Prove the model solved the task

Report three complementary metrics:

- **collision rate** — did any waypoint enter the obstacle?
- **endpoint error** — did the chunk terminate at the commanded goal?
- **mode balance** — did the sampler preserve both upper and lower solutions?

A low loss alone is not evidence of a useful policy.

In [ ]:
def evaluate(samples, cond):
    x = X_GRID.view(1, -1)
    radii = cond[:, 1].view(-1, 1)
    distances = torch.sqrt(x.square() + samples.square())
    collision = (distances < radii).any(dim=1)
    endpoint_error = (samples[:, -1] - cond[:, 0]).abs()
    upper_mode = samples[:, N_POINTS // 2] > 0
    return {
        "collision_rate": float(collision.float().mean()),
        "mean_endpoint_error": float(endpoint_error.mean()),
        "upper_mode_fraction": float(upper_mode.float().mean()),
    }


metrics = evaluate(samples, cond_test)
for name, value in metrics.items():
    print(f"{name:24s}: {value:.3f}")

# The conditional-mean baseline is the straight line between the endpoints.
s = torch.linspace(0.0, 1.0, N_POINTS, device=DEVICE).view(1, -1)
regression_baseline = s * cond_test[:, 0:1]
baseline_metrics = evaluate(regression_baseline, cond_test)
print("\nconditional-mean baseline")
for name, value in baseline_metrics.items():
    print(f"{name:24s}: {value:.3f}")

# Loose workshop checks: exact values vary with hardware, but the qualitative result should not.
assert metrics["collision_rate"] < 0.20, "Train longer or inspect the denoising implementation."
assert metrics["mean_endpoint_error"] < 1e-5, "The hard endpoint projection is broken."
assert 0.20 < metrics["upper_mode_fraction"] < 0.80, "The sampler dropped a mode."
assert baseline_metrics["collision_rate"] > 0.95, "The baseline should expose the averaging failure."
print("\nworkshop checks passed")

In [ ]:
# Final comparison figure for the turn-in.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharex=True, sharey=True)

for y, color in [(upper, "#0f766e"), (lower, "#2563eb")]:
    axes[0].plot(torch.linspace(-1, 1, N_POINTS), y, color=color, lw=3)
draw_obstacle(axes[0], 0.23)
axes[0].set_title("Expert demonstrations")

axes[1].plot(torch.linspace(-1, 1, N_POINTS), regression_baseline[0].cpu(), color="#dc2626", lw=3)
draw_obstacle(axes[1], 0.23)
axes[1].set_title("Conditional mean")

for y in samples.detach().cpu():
    axes[2].plot(torch.linspace(-1, 1, N_POINTS), y, color="#0f766e", alpha=0.18, lw=2)
draw_obstacle(axes[2], 0.23)
axes[2].set_title("Diffusion samples")

for ax in axes:
    ax.set(xlim=(-1.05, 1.05), ylim=(-0.85, 0.85), xlabel="x")
axes[0].set_ylabel("trajectory y")
plt.tight_layout()
plt.show()

## 7 · Stress tests and discussion

Choose at least one:

1. **Remove multimodality.** Train only upper demonstrations. What changes in mode coverage and collision rate?
2. **Make chunks longer.** Double **N_POINTS**. Does training or sampling become harder?
3. **Reduce denoising steps.** Sample with fewer reverse steps. Where does quality fail first?
4. **Distribution shift.** Evaluate obstacle radii outside the training range.
5. **RTC bridge.** Pretend inference consumed the first $k$ actions: freeze that prefix and regenerate only the overlapping tail.

Turn in one figure containing expert paths, the MSE baseline, and diffusion samples, plus three sentences explaining what the generative model learned that regression could not.

### Sources

- Ho, Jain & Abbeel, *Denoising Diffusion Probabilistic Models* (2020), https://arxiv.org/abs/2006.11239
- Chi et al., *Diffusion Policy: Visuomotor Policy Learning via Action Diffusion* (2023), https://arxiv.org/abs/2303.04137